<a href="https://colab.research.google.com/github/guzonghua/saavlabs/blob/main/Lab3/Lab3_UFLD_Lane_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC144V Lab 3: Lane Detection with UFLD and Adversarial Attacks

Read the handout first. Group name: **TODO**   Members: **TODO**

Runtime tip: Task 3 needs many forward/backward passes, so use *Runtime > Change runtime type > GPU*.

## Setup

Run this cell once. It needs `ufld_lab.py` and `lab3_data.zip` from Canvas in the working directory (upload them with the Colab file browser).

In [ ]:
import os, glob, time, zipfile
import numpy as np, cv2, torch, torch.nn.functional as F
import matplotlib.pyplot as plt

if not os.path.exists('Ultra-Fast-Lane-Detection'):
    !git clone https://github.com/cfzd/Ultra-Fast-Lane-Detection
if not os.path.exists('tusimple_18.pth'):
    !pip -q install gdown
    !gdown 1WCYyur5ZaWczH15ecmeDowrW30xcLrCn -O tusimple_18.pth   # if this fails (quota), use the copy on Canvas
if not os.path.exists('lab3_data') and os.path.exists('lab3_data.zip'):
    zipfile.ZipFile('lab3_data.zip').extractall('.')
assert os.path.exists('ufld_lab.py'), 'upload ufld_lab.py from Canvas first'

from ufld_lab import LaneDetector, draw_lanes, video_frames, cell_stats, NET_H, NET_W

det = LaneDetector('Ultra-Fast-Lane-Detection', 'tusimple_18.pth', 'Tusimple')
print('device:', det.device, '| grid columns G =', det.G, '| row anchors A =', det.A)
FRAMES = sorted(glob.glob('lab3_data/frames/*.jpg')) + sorted(glob.glob('lab3_data/frames/*.png'))
VIDEOS = sorted(glob.glob('lab3_data/videos/*.mp4'))
print(len(FRAMES), 'frames,', len(VIDEOS), 'videos')

## Task 1: Run the pretrained detector (20 points)

A worked example on one frame:

In [ ]:
img = cv2.imread(FRAMES[0])                      # BGR image, any size
lanes, logits = det.detect(img)                   # lanes in ORIGINAL image coordinates
print('logits shape:', tuple(logits.shape), '| points per lane slot:', [len(l) for l in lanes])
plt.figure(figsize=(8, 4.5)); plt.imshow(cv2.cvtColor(draw_lanes(img, lanes), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

In [ ]:
# TODO 1.1: show at least 6 frames with detected lanes in a grid
# TODO 1.2: pick 2 failure cases and show them (explain the cause in the answers cell below)
# TODO 1.3: time pre-processing (det.to_tensor), network (det.logits) and decoding (det.decode) over 100 runs;
#           report ms and FPS (on CPU, and on GPU if available; call torch.cuda.synchronize() before reading the clock)


**Task 1 answers** (a) output tensor dimensions; (b) row-anchor classification vs. segmentation; (c) class G; (d) grid cell width in pixels; plus your failure-case analysis and timing table:

TODO

## Task 2: Lane departure warning (35 points)

### 2.1 `lateral_offset`
Follow the specification in the handout (section 5.1).

In [ ]:
def lateral_offset(lanes, img_w, img_h, y_ref=0.9, lane_width_m=3.7, deg=2):
    """Return None if the ego lane cannot be determined, else a dict with at least
    offset_px, offset_m (positive = vehicle right of the lane centre), xl, xr, width_px."""
    # TODO: 1) fit x = f(y) per lane (np.polyfit / np.polyval, only lanes with >= 3 points)
    #       2) evaluate at y = y_ref * img_h
    #       3) ego lane = closest line left / right of the image centre; None if a side is missing
    #       4) offset_px = img_w/2 - (xl + xr)/2 ;  offset_m = offset_px * lane_width_m / (xr - xl)
    raise NotImplementedError

In [ ]:
# quick check on one frame: draw the two ego lines at y_ref and the lane centre
img = cv2.imread(FRAMES[0]); h, w = img.shape[:2]
lanes, _ = det.detect(img)
r = lateral_offset(lanes, w, h)
print(r)
vis = draw_lanes(img, lanes)
if r is not None:
    y = int(0.9 * h)
    for x_, col in ((r['xl'], (255, 0, 0)), (r['xr'], (0, 0, 255)), ((r['xl'] + r['xr']) / 2, (0, 255, 0)), (w / 2, (255, 255, 0))):
        cv2.line(vis, (int(x_), y - 40), (int(x_), y + 40), col, 3)
plt.figure(figsize=(8, 4.5)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

### 2.2 Time series and warning

In [ ]:
MAX_FRAMES = 300            # use fewer while debugging
VIDEO = VIDEOS[0]
offs = []
for i, frame in video_frames(VIDEO, stride=1, max_frames=MAX_FRAMES):
    lanes, _ = det.detect(frame)
    r = lateral_offset(lanes, frame.shape[1], frame.shape[0])
    offs.append(np.nan if r is None else r['offset_m'])
offs = np.array(offs)
print(len(offs), 'frames,', int(np.isnan(offs).sum()), 'frames without ego lane')

In [ ]:
def ema(series, alpha=0.3):
    """Exponential moving average; NaN entries (lane lost) keep the previous value."""
    # TODO
    raise NotImplementedError

def ldw_warning(offset_m, thresh_m=0.6, min_frames=3):
    """Boolean array: True where |offset| > thresh_m for at least min_frames consecutive frames."""
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: plot raw and EMA-smoothed offsets (alpha in {0.1, 0.3, 0.5}) versus time, mark lost-lane frames,
#       count warning events, show 3 frames where a warning triggers, report jitter = std(np.diff(signal))


**Task 2 answers**: assumptions and failure cases of your estimate, effect of smoothing (jitter vs. lag), warning events:

TODO

## Task 3: Adversarial attacks on the lane detector (40 points)

White-box, digital, L-infinity. Work on the 288 x 800 network input `x` in [0, 1]; labels are the model's own clean predictions. Remember: `det.forward` is differentiable, `det.logits` is not.

In [ ]:
def clean_labels(logits):
    """Pseudo-labels: the clean prediction, shape (1, A, 4), values 0..G (G = 'no lane')."""
    return logits.argmax(dim=1)

def shift_target(y_clean, shift_cells, G):
    """Target labels: every lane cell moved `shift_cells` columns to the right; no-lane cells unchanged."""
    # TODO
    raise NotImplementedError

def fgsm(det, x, y, eps):
    """Untargeted FGSM."""
    # TODO
    raise NotImplementedError

def pgd_linf(det, x, y, eps, alpha, steps, targeted=False):
    """L-inf PGD. Untargeted: ascend CE(f(x_adv), y). Targeted: descend CE(f(x_adv), y_target)."""
    # TODO (project onto the eps-ball around x AND onto [0, 1] after every step)
    raise NotImplementedError

In [ ]:
# frames used for the experiments (the handout asks for at least 10)
N_ATTACK_FRAMES = 10                       # use 2 while debugging
att_frames = FRAMES[:N_ATTACK_FRAMES]
XS = [det.to_tensor(cv2.imread(f)) for f in att_frames]

def offset_of(logits):
    """Lateral offset in metres from logits (decoded at network resolution; metres do not depend on resolution)."""
    r = lateral_offset(det.lanes_at_net_res(logits), NET_W, NET_H)
    return np.nan if r is None else r['offset_m']

def run_attack(x, kind, eps, s=None, steps=20):
    """Returns (x_adv, logits_clean, logits_adv, y_target) for one frame."""
    lg0 = det.logits(x); yc = clean_labels(lg0); yt = None
    if kind == 'fgsm':
        xa = fgsm(det, x, yc, eps)
    elif kind == 'pgd':
        xa = pgd_linf(det, x, yc, eps, eps / 4, steps)
    elif kind == 'pgd_targeted':
        yt = shift_target(yc, s, det.G)
        xa = pgd_linf(det, x, yt, eps, eps / 10, steps, targeted=True)
    return xa, lg0, det.logits(xa), yt

In [ ]:
# TODO: for FGSM (eps 2,4,8,16 /255), PGD (same eps) and targeted PGD (s = +2,+4,+6, eps as needed, up to 100 steps)
#       over all frames, collect: % cells changed, mean shift (px), target hit rate (targeted only),
#       mean |change in offset| (m) and whether the warning decision (|offset| > 0.6 m) flips.
#       Print a table like the one in the handout.


In [ ]:
# TODO: visualise one frame: original | amplified perturbation | attacked, lanes drawn on original and attacked


**Task 3 answers** (handout section 6.4): (a) most effective attack per unit of eps; (b) effect of a lane shift on a lane-keeping controller; (c) digital vs. physical attacks; (d) one defence and its expected effect.

TODO

## Submission checklist

- Notebook executed, all outputs visible, sharing set to *anyone with the link*
- PDF report (at most 6 pages) with figures, tables and answers for Tasks 1 to 3